# Case 4: Çok Katmanlı Anomali Tespiti Defteri

Case brief'in istediği mimari: tek bir modele bağlı kalmadan, **column** (kolon bazlı), **multivariate** (çok değişkenli), **entity** (varlık bazlı) ve **temporal** (zaman bazlı) katmanlarının her biri kendi bakış açısına göre ayrı bir anomali skoru üretiyor. Her katman `src/services/anomaly/` altında ayrı bir dosya, `TransactionID` + skor kolonları üreten bir fonksiyon olarak yazılıyor; Case 3'ün feature katmanlarıyla aynı desen. Bu notebook, katmanları sırayla ekliyor.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found: expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import pandas as pd
import pyarrow.parquet as pq

from src.config import settings

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

parquet_path = settings.processed_data_path / "merged_transactions.parquet"
print(f"kaynak: {parquet_path}")

kaynak: /home/canberk/workspace/case_study/data/processed/merged_transactions.parquet


## 1. Column Anomali Tespiti

Her sayısal kolon için, bir işlemin değerinin o kolonun tipik değerinden ne kadar saptığını **MAD tabanlı (robust) z-score** ile ölçüyoruz; ham z-score değil, çünkü Case 1, 96 sayısal kolonun 91'inin aşırı sağa çarpık olduğunu bulmuştu; ortalama/standart sapma bu çarpıklıkta yanıltıcı olur. `distributions.py`'nin zaten önerdiği kolonlarda önce `log1p` uygulanıyor; burada yeni bir karar verilmiyor, Case 1'in doğrulanmış kararı yeniden kullanılıyor.

96 kolonun skorları tek bir satır skoruna dört sinyalle indirgeniyor:
- **`column_anomaly_mean_abs_zscore`**: bu katmanın ana skoru, satırın skorlanan kolonlar genelinde ortalama anomallik derecesi.
- **`column_anomaly_scored_count`**: bu satırda kaç kolonun gerçekten skorlanabildiği (aşağıda önemli bir bulgu bunun etrafında).
- **`column_anomaly_extreme_column_count`**: kaç kolon tek başına eşiği (`MAD_OUTLIER_THRESHOLD=3.5`, `quality.py`'den) aşıyor.
- **`column_anomaly_top_column`** / **`column_anomaly_max_abs_zscore`**: açıklanabilirlik çapası, satırın skoruna en çok hangi tek kolon sebep oluyor.

In [3]:
from src.services.anomaly.column_anomaly import compute_column_anomaly_scores

column_anomaly = compute_column_anomaly_scores(parquet_path)
print(f"şekil: {column_anomaly.shape}")
column_anomaly.head(10)

şekil: (590540, 6)


,TransactionID,column_anomaly_mean_abs_zscore,column_anomaly_scored_count,column_anomaly_extreme_column_count,column_anomaly_max_abs_zscore,column_anomaly_top_column
0,2987000,0.226486,3,0,0.674500,C13
1,2987001,0.970291,3,0,1.153065,C9
2,2987002,0.289373,3,0,0.674500,C13
3,2987003,0.741222,3,0,1.821447,C13
4,2987004,0.688581,4,0,1.153065,C9
5,2987005,0.367379,3,0,0.674500,C13
6,2987006,0.579943,3,0,1.065328,TransactionAmt
7,2987007,1.380759,3,0,2.314711,TransactionAmt
8,2987008,0.993446,4,0,1.890160,TransactionAmt
9,2987009,0.991504,3,0,1.153065,C9


### Beklenmedik bulgu: skor aslında sadece 8 kolondan geliyor

`column_anomaly_scored_count`'ın dağılımına bakalım.

In [4]:
column_anomaly["column_anomaly_scored_count"].describe()

count    590540.000000
mean          3.509728
std           1.050622
min           3.000000
25%           3.000000
50%           3.000000
75%           3.000000
max           8.000000
Name: column_anomaly_scored_count, dtype: float64

Medyan **3**, maksimum **8**: 96 sayısal kolonun neredeyse tamamı bir satırda hiç katkı vermiyor. Sebebini araştıralım: fonksiyon, `MAD=0` çıkan kolonları (log1p sonrası bile) atlıyor; bir kolonun medyanının etrafındaki mutlak sapmaların medyanı sıfırsa (yani değerlerin **yarısından fazlası tek bir değere eşitse**), o kolonda "tipik sapma" diye bir şey tanımlanamaz.

In [5]:
import numpy as np
from src.services.analyzers.column_types import profile_columns, classify_columns
from src.services.analyzers.distributions import compute_numeric_distributions, NUMERIC_SEMANTIC_TYPES

profile = profile_columns(parquet_path)
classified = classify_columns(profile)
numeric_cols = classified[classified["semantic_type"].isin(NUMERIC_SEMANTIC_TYPES)]["column"].tolist()
dist = compute_numeric_distributions(parquet_path, numeric_cols)
log_cols = set(dist.loc[dist["log_transform_recommended"], "column"])

pf = pq.ParquetFile(parquet_path)
mad_zero_cols, mad_nonzero_cols = [], []
for col in numeric_cols:
    values = pf.read(columns=[col]).to_pandas()[col].to_numpy(dtype=np.float64)
    if col in log_cols:
        values = np.log1p(np.clip(values, a_min=0, a_max=None))
    valid = ~np.isnan(values)
    if valid.sum() == 0:
        continue
    median = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values[valid] - median))
    (mad_zero_cols if mad == 0 else mad_nonzero_cols).append(col)

print(f"toplam sayısal kolon: {len(numeric_cols)}")
print(f"MAD=0 (atlanan) kolon sayısı: {len(mad_zero_cols)}")
print(f"MAD!=0 (fiilen skorlanan) kolon sayısı: {len(mad_nonzero_cols)}")
print(f"skorlanan kolonlar: {mad_nonzero_cols}")

toplam sayısal kolon: 96
MAD=0 (atlanan) kolon sayısı: 88
MAD!=0 (fiilen skorlanan) kolon sayısı: 8
skorlanan kolonlar: ['TransactionAmt', 'C9', 'C13', 'D8', 'D9', 'id_02', 'id_21', 'id_25']


**Yorum:** Case 1'in `near_constant` sınıflandırması %99 baskın-değer eşiği kullanıyordu. MAD=0 için gereken eşik çok daha düşük: sadece **%50+**. Yani birçok `C` (sayaç) ve `V` kolonu, `near_constant` sayılacak kadar aşırı olmasa da (%99'un altında), MAD'yi sıfırlayacak kadar (yarısından fazlası tek değer, genelde 0) sabit. Bu, robust istatistiklerin bu veri setinde beklenenden daha erken "tükenmesi"; dürüstçe raporlanması gereken bir sınırlama.

**Sonuç, gizlenmiyor:** bu katmanın gerçek ayırt ediciliği sadece 8 kolondan geliyor (`TransactionAmt`, `C9`, `C13`, `D8`, `D9`, `id_02`, `id_21`, `id_25`). Bu tam olarak Case 4'ün **çok katmanlı** mimari istemesinin sebebi: tek bir istatistiksel yaklaşım (burada: MAD tabanlı column anomaly) veri setinin büyük kısmını göremiyor; multivariate/entity/temporal katmanlar bu boşluğu farklı açılardan kapatacak.

### Neden MAD, başka ne olabilirdi?

Ham z-score (`(değer-ortalama)/std`) kullanmadık çünkü ortalama/std, birkaç uç değerden kendisi de sürükleniyor; Case 1'in bulduğu aşırı çarpık kolonlarda (kurtosis 150.000+) bu, normal değerleri anomali gibi gösterebilirdi. Medyan/MAD, çoğunluğun yoğunlaştığı yerden kolayca sürüklenmediği için tercih edildi.

**Alternatifler:**
- **IQR/Tukey fences**: `quality.py`'de zaten var, aynı dayanıklılık ailesinden; burada tek kompozit skora karmaşıklık katmamak için kullanılmadı.
- **Percentile-rank (ECDF) skoru**: yayılma ölçüsüne ihtiyaç duymaz, bugünkü 88-kolonluk atlama sorununu muhtemelen hiç yaşamazdık; geriye dönük bakınca bu veri setinde daha iyi bir ilk seçim olabilirdi.
- **KDE / rank-then-zscore / Isolation Forest**: sırasıyla, hesaplama maliyeti yüksek ve aynı seyreklik sorununu yaşayabilir; log-dönüşüm kararını otomatikleştirir; çok-değişkenli olduğu için buraya değil sıradaki multivariate katmana daha uygun.

### En yüksek skorlu 10 işlem

In [6]:
column_anomaly.sort_values("column_anomaly_mean_abs_zscore", ascending=False).head(10)

,TransactionID,column_anomaly_mean_abs_zscore,column_anomaly_scored_count,column_anomaly_extreme_column_count,column_anomaly_max_abs_zscore,column_anomaly_top_column
340779,3327779,26.222767,5,1,127.255667,id_21
280104,3267104,25.198229,5,2,114.665000,id_21
254892,3241892,25.032036,6,3,134.225500,id_21
14099,3001099,24.665642,6,2,134.000667,id_21
170821,3157821,24.618388,6,3,134.000667,id_21
73089,3060089,24.613388,6,3,134.225500,id_21
412190,3399190,24.550857,6,2,134.225500,id_21
383695,3370695,24.547679,5,2,114.665000,id_21
545290,3532290,24.469087,6,2,134.225500,id_21
191013,3178013,24.463441,6,3,134.000667,id_21


Not: `column_anomaly_scored_count` sütununa dikkat, en tepedeki satırların çoğunda sadece 5-6 kolon skorlanmış. Az sayıda kolona dayanan bir ortalama, tek bir uç değerden aşırı etkilenir; bu yüzden `mean_abs_zscore`'u tek başına "kesin anomali" gibi okumak yerine, `scored_count` ile birlikte değerlendirmek gerekiyor; düşük `scored_count`'lu yüksek skorlar daha temkinli yorumlanmalı.

### Betimleyici kontrol: skor ile fraud oranı ilişkili mi?

In [7]:
isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()
check = column_anomaly.merge(isfraud, on="TransactionID")
check["score_bucket"] = pd.qcut(check["column_anomaly_mean_abs_zscore"], 5, labels=["1 (en düşük)", "2", "3", "4", "5 (en yüksek)"])

check.groupby("score_bucket", observed=True)["isFraud"].agg(fraud_rate_pct=lambda s: s.mean() * 100, count="count")

,fraud_rate_pct,count
score_bucket,,
1 (en düşük),1.714373,118119
2,2.308272,118097
3,3.683168,118132
4,5.335896,118087
5 (en yüksek),4.453664,118105


En düşük skor grubunda fraud oranı %1,71, 4. grupta %5,34'e çıkıyor: yaklaşık 3 kat, anlamlı bir ayrım. 5. (en yüksek) grupta %4,45'e hafifçe geriliyor; muhtemelen yukarıdaki `scored_count`-kaynaklı volatilite etkisi (çok yüksek skorların bir kısmı az-kolonlu, gürültülü satırlardan geliyor). Bu küçük tutarsızlık gizlenmiyor; katmanın genel eğilimi güçlü ama en uç dilimde biraz gürültülü.

## 2. Multivariate Anomali Tespiti

Column anomaly kolonları **tek tek** değerlendiriyordu. Bu katman, birden fazla kolonu **birlikte** değerlendiriyor; bir işlem her kolonda ayrı ayrı normal görünse de, kolonların **kombinasyonu** anormal olabilir (örn. hem tutar hem sayaç değeri tek başına sıra dışı değil ama birlikte görülmeleri nadir).

### Kolon kümesi araştırması

`column_anomaly.py`'nin bulduğu 8 "sağlıklı" (MAD≠0) kolonu doğrudan kullanmayı denedik ama bir sorun çıktı: bu 8 kolonun **hepsinin aynı anda dolu olduğu** satır sayısını kontrol edelim.

In [8]:
candidate_cols = ["TransactionAmt", "C9", "C13", "D8", "D9", "id_02", "id_21", "id_25"]
df_check = pq.ParquetFile(parquet_path).read(columns=candidate_cols).to_pandas()

print("her kolonun ayrı eksik oranı (%):")
print((df_check.isna().mean() * 100).round(2))

complete = df_check.dropna()
print(f"\n8 kolonun HEPSİ dolu olan satır: {len(complete)} (%{len(complete)/len(df_check)*100:.2f})")

her kolonun ayrı eksik oranı (%):
TransactionAmt     0.00
C9                 0.00
C13                0.00
D8                87.31
D9                87.31
id_02             76.15
id_21             99.13
id_25             99.13
dtype: float64

8 kolonun HEPSİ dolu olan satır: 2971 (%0.50)


**Sadece %0,5.** `id_02`/`D8`/`D9`/`id_21`/`id_25`, %76-99 arası eksik; hepsini birlikte isteyince neredeyse hiçbir satır kalmıyor. Eksiklik eşiğini %30'a kadar gevşetsek bile durum değişmiyor: bu veri setinde %0 ile %76 arasında "orta düzeyde eksik ve MAD≠0" bir kolon yok, boşluk sert.

**Karar:** yalnızca hem MAD≠0 hem **tamamen dolu** (%0 eksik) olan üç kolonla ilerliyoruz: `TransactionAmt`, `C9`, `C13`. İnce bir özellik uzayı ama her satır gerçekten skorlanabiliyor; column anomaly'nin aksine (kolon bazlı, satır satır eksik toleranslı), multivariate bir mesafe ölçümü tüm boyutların o satırda dolu olmasını gerektiriyor.

### İki yöntem, aynı 3 kolon

- **Mahalanobis distance** (saf numpy): çok değişkenli merkeze, kolonlar arası korelasyonu hesaba katarak uzaklık. Kabaca elips-şekilli dağılım varsayıyor. Her kolonun toplam uzaklığa katkısı doğrudan hesaplanabiliyor (açıklanabilirlik).
- **Isolation Forest** (`scikit-learn`): ağaç tabanlı, doğrusal olmayan yapıyı da yakalayabiliyor. Mahalanobis kadar doğrudan ayrıştırılamıyor; açıklama için her satırın tek-kolon bazında en büyük MAD sapmasını (IF'in kendi iç mantığı değil, pratik bir yaklaşık gösterge) raporluyoruz.

In [9]:
from src.services.anomaly.multivariate_anomaly import compute_multivariate_anomaly_scores

multivariate = compute_multivariate_anomaly_scores(parquet_path)
print(f"şekil: {multivariate.shape}")
multivariate.head(10)

şekil: (590540, 5)


,TransactionID,multivariate_mahalanobis_distance,multivariate_mahalanobis_top_contributor,multivariate_isolation_forest_score,multivariate_isolation_forest_top_contributor
0,2987000,1.092386,C13,-0.109392,C13
1,2987001,1.358261,TransactionAmt,-0.095853,C9
2,2987002,1.147110,C13,-0.115453,C13
3,2987003,1.769160,C13,-0.052951,C13
4,2987004,0.996790,C9,-0.115155,C9
5,2987005,1.240191,C13,-0.111240,C13
6,2987006,1.205895,C13,-0.090468,TransactionAmt
7,2987007,2.036900,TransactionAmt,-0.000230,TransactionAmt
8,2987008,1.924905,TransactionAmt,-0.036925,TransactionAmt
9,2987009,0.751184,C9,-0.049044,C9


### Doğrulama: Mahalanobis, scipy'nin bağımsız implementasyonuyla eşleşiyor mu?

In [10]:
from scipy.spatial.distance import mahalanobis
from src.services.anomaly.multivariate_anomaly import MULTIVARIATE_COLUMNS, _prepare_features

df = _prepare_features(parquet_path)
X = df[MULTIVARIATE_COLUMNS].to_numpy()
mean = X.mean(axis=0)
inv_cov = np.linalg.pinv(np.cov(X, rowvar=False))

for i in range(5):
    scipy_dist = mahalanobis(X[i], mean, inv_cov)
    our_dist = multivariate["multivariate_mahalanobis_distance"].iloc[i]
    print(f"satır {i}: scipy={scipy_dist:.6f}  bizim={our_dist:.6f}  fark={abs(scipy_dist-our_dist):.2e}")

satır 0: scipy=1.092386  bizim=1.092386  fark=0.00e+00
satır 1: scipy=1.358261  bizim=1.358261  fark=0.00e+00
satır 2: scipy=1.147110  bizim=1.147110  fark=0.00e+00
satır 3: scipy=1.769160  bizim=1.769160  fark=0.00e+00
satır 4: scipy=0.996790  bizim=0.996790  fark=0.00e+00


Tam eşleşme (fark 0): bağımsız bir kütüphaneyle çapraz doğrulandı.

### İki yöntem ne kadar örtüşüyor?

In [11]:
rank_corr = multivariate[["multivariate_mahalanobis_distance", "multivariate_isolation_forest_score"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman sıra korelasyonu: {rank_corr:.3f}")

Spearman sıra korelasyonu: 0.758


**0,76: güçlü ama mükemmel değil.** İki yöntemin aynı fikirde olmadığı yer, tam olarak beklediğimiz yer: Mahalanobis'in elips-şekilli varsayımının kırıldığı, doğrusal olmayan bölgeler. Bu ayrışma bir kusur değil; Case 4'ün "tek modele bağlı kalmama" ilkesinin bu katmanın **içinde** bile geçerli olduğunu gösteriyor.

### Betimleyici kontrol: her iki skor da fraud oranıyla ilişkili mi?

In [12]:
isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()
check = multivariate.merge(isfraud, on="TransactionID")

for col in ["multivariate_mahalanobis_distance", "multivariate_isolation_forest_score"]:
    check[f"{col}_bucket"] = pd.qcut(check[col], 5, labels=["1 (düşük)", "2", "3", "4", "5 (yüksek)"], duplicates="drop")
    print(f"--- {col} ---")
    print((check.groupby(f"{col}_bucket", observed=True)["isFraud"].mean() * 100).round(2))
    print()

--- multivariate_mahalanobis_distance ---
multivariate_mahalanobis_distance_bucket
1 (düşük)     1.83
2             2.25
3             4.06
4             4.55
5 (yüksek)    4.85
Name: isFraud, dtype: float64

--- multivariate_isolation_forest_score ---
multivariate_isolation_forest_score_bucket
1 (düşük)     2.00
2             2.11
3             2.38
4             5.13
5 (yüksek)    5.87
Name: isFraud, dtype: float64



İkisi de tutarlı ve monotonik bir artış gösteriyor: Mahalanobis %1,83→%4,85, Isolation Forest %2,00→%5,87 (en yüksek dilimde biraz daha güçlü ayrım). Column anomaly'de gördüğümüz en-uç-dilimde-hafif-düşüş sorunu burada yok; muhtemelen bu katmanın her satırı aynı 3 tam-dolu kolona dayandığı için, column anomaly'nin `scored_count` volatilitesi burada söz konusu değil.

## 3. Entity Anomali Tespiti

Önceki iki katman veriyi **global popülasyona** göre değerlendiriyordu (bir kolonun/kombinasyonun tüm veri setindeki tipik değerine göre). Bu katman farklı bir soru soruyor: **bu işlem, aynı kartın (`card1`) kendi geçmişine göre normal mi?**

Yeni bir hesaplama yazmıyoruz; Case 3'te zaten inşa edilen nedensel (`_so_far`) feature'ları yeniden kullanıyoruz: `entity.py`'den `user_amount_zscore` (tutar sapması), `relational.py`'den `is_new_addr1_for_card`/`is_new_device_for_card` (davranış yeniliği). Case 3 girdileri üretmişti, Case 4 bunları tek bir açıklanabilir skora dönüştürüyor.

**Skor = |tutar sapması| + yenilik bileşeni.** Yenilik bileşeni (`is_new_addr1_for_card` + `is_new_device_for_card`, eşit ağırlıkla) sadece kartın **zaten bir geçmişi varsa** sayılıyor; yepyeni bir kart için her adres/cihaz zaten "yeni"dir, bu triviyal durumda hiçbir şey ayırt etmez. Ağırlıklar (`NEW_ADDR_WEIGHT=NEW_DEVICE_WEIGHT=1,0`) `isFraud`'a göre ayarlanmadı; sadece tipik bir z-score ile aynı ölçekte olacak şekilde seçildi (bu veri setinde medyan |z|≈0,43): nedensellik/etiket kalibrasyonu değil, ölçek kalibrasyonu.

In [13]:
from src.services.anomaly.entity_anomaly import compute_entity_anomaly_scores

entity_anomaly = compute_entity_anomaly_scores(parquet_path)
print(f"şekil: {entity_anomaly.shape}")
entity_anomaly.head(10)

şekil: (590540, 5)


,TransactionID,entity_anomaly_score,entity_anomaly_history_depth,entity_anomaly_amount_component,entity_anomaly_novelty_component
0,3230924,0.000000,0,0.000000,0.0
1,3023634,0.000000,0,0.000000,0.0
2,3151336,0.000000,1,0.000000,0.0
3,3210739,0.725473,2,0.725473,0.0
4,3020767,0.000000,0,0.000000,0.0
5,3028973,2.000000,1,0.000000,2.0
6,3386444,2.471405,2,0.471405,2.0
7,3504371,2.322772,3,2.322772,0.0
8,3504379,1.227095,4,1.227095,0.0
9,3038871,0.000000,0,0.000000,0.0


### Column anomaly'de gördüğümüz aynı ders burada da geçerli

En yüksek skorlu işlemlere bakalım; `entity_anomaly_history_depth` sütununa dikkat.

In [14]:
entity_anomaly.sort_values("entity_anomaly_score", ascending=False).head(10)

,TransactionID,entity_anomaly_score,entity_anomaly_history_depth,entity_anomaly_amount_component,entity_anomaly_novelty_component
112361,3041633,8028.136840,2,8028.136840,0.0
259013,3211281,4865.601761,2,4865.601761,0.0
161665,3116658,3558.902529,3,3558.902529,0.0
460187,3262758,1572.551754,2,1571.551754,1.0
316819,2994184,1472.489212,2,1471.489212,1.0
238224,3084773,1131.734505,2,1131.734505,0.0
513891,2990825,840.335749,2,839.335749,1.0
471553,3339482,743.776236,4,742.776236,1.0
117634,3324366,733.280838,10,733.280838,0.0
28916,3116402,713.670783,2,712.670783,1.0


Tepedeki tüm satırlar 2-10 arası çok sığ bir geçmişe (`history_depth`) dayanıyor; az sayıda önceki işlemden hesaplanan bir ortalama/sapma, tek bir uç değerden aşırı etkileniyor. Bu, `column_anomaly`'deki `scored_count` dersiyle birebir aynı: **az örneğe dayanan bir skor daha gürültülü olur**, katmandan katmana tekrar eden bir prensip.

In [15]:
import numpy as np

corr = np.corrcoef(
    np.log1p(entity_anomaly["entity_anomaly_history_depth"]),
    np.log1p(entity_anomaly["entity_anomaly_score"]),
)[0, 1]
print(f"log(history_depth) ile log(score) korelasyonu: {corr:.3f}")

log(history_depth) ile log(score) korelasyonu: -0.169


Negatif korelasyon (-0,17): sığ geçmiş, sistematik olarak daha yüksek/gürültülü skor üretiyor. Bunu gizlemek yerine `entity_anomaly_history_depth`'i çıktının bir parçası yaptık ki bu skoru kullanan biri düşük-derinlikli yüksek skorlara temkinli yaklaşsın.

### Betimleyici kontrol: skor ile fraud oranı, ve ilginç bir "soğuk başlangıç" bulgusu

In [16]:
isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()
check = entity_anomaly.merge(isfraud, on="TransactionID")

has_history = check[check["entity_anomaly_history_depth"] > 0].copy()
has_history["bucket"] = pd.qcut(has_history["entity_anomaly_score"], 5, labels=["1", "2", "3", "4", "5"], duplicates="drop")
print("fraud oranı (geçmişi olan işlemler, skor kovalarına göre):")
print((has_history.groupby("bucket", observed=True)["isFraud"].mean() * 100).round(2))

first_txn_fraud = check.loc[check["entity_anomaly_history_depth"] == 0, "isFraud"].mean() * 100
overall_fraud = check["isFraud"].mean() * 100
print(f"\nilk işlemlerin (skor her zaman 0, geçmiş yok) fraud oranı: %{first_txn_fraud:.2f}")
print(f"genel fraud oranı: %{overall_fraud:.2f}")

fraud oranı (geçmişi olan işlemler, skor kovalarına göre):
bucket
1    3.19
2    2.90
3    2.89
4    3.88
5    4.75
Name: isFraud, dtype: float64



ilk işlemlerin (skor her zaman 0, geçmiş yok) fraud oranı: %2.57
genel fraud oranı: %3.50


**İki bulgu, ikisi de dürüstçe raporlanıyor:**

1. Skor kovaları arasındaki ilişki genel olarak yükseliyor (%3,19 → %4,75) ama **monotonik değil**: 2. ve 3. kova geriliyor (%2,90, %2,89). Diğer katmanlara göre daha gürültülü; bu, yukarıdaki sığ-geçmiş etkisiyle tutarlı.
2. **Soğuk başlangıç (yepyeni kart, `history_depth=0`) işlemlerinde fraud oranı %2,57: genel ortalamanın (%3,50) altında.** "Geçmişi olmayan kart = şüpheli" gibi sezgisel bir varsayım burada da (Case 3'teki `is_new_addr1_for_card` bulgusuna benzer şekilde) doğrulanmıyor. Bu yüzden bu katmanda soğuk-başlangıç işlemlerine skor 0 vermek ("bilgi yok" anlamında), onları otomatik olarak yüksek şüpheli saymaktan ("yeni=riskli" varsayımı) daha isabetli bir tasarım tercihi olarak doğrulanmış oluyor.

## 4. Temporal Anomali Tespiti

Case 4'ün son katmanı: işlemin **zamanına** göre ne kadar sıra dışı olduğu. İki bileşen:

- **Tutar bileşeni**: Case 3'ün `context.py`'sinden doğrudan yeniden kullanılıyor (`amount_zscore_within_hour`); bu tutar, aynı saat diliminin (`hour_of_day`) kendi geçmişine göre ne kadar sapıyor.
- **Burst (patlama) bileşeni**: burada yeni; işlemler gerçek (gün, saat) bloklarına ayrılıyor (182 gün × 24 saat = 4368 blok, saat başına \~182 gözlem, istikrarlı istatistik için yeterli), her bloğun işlem sayısı **aynı saat_of_day'i paylaşan diğer bloklara göre** skorlanıyor (Case 1'in bulduğu \~15 kata varan saatlik hacim farkını hesaba katarak: saat 3'teki bir patlama, saat 18'in tipik hacmiyle değil saat 3'ün tipik hacmiyle karşılaştırılıyor). Sadece **pozitif** yön sayılıyor (sakin bir saat "anomali" sayılmıyor, sadece beklenmedik yoğunluk artışı).

In [17]:
from src.services.anomaly.temporal_anomaly import compute_temporal_anomaly_scores

temporal_anomaly = compute_temporal_anomaly_scores(parquet_path)
print(f"şekil: {temporal_anomaly.shape}")
temporal_anomaly.head(10)

şekil: (590540, 6)


,TransactionID,temporal_anomaly_score,temporal_anomaly_amount_component,temporal_anomaly_burst_component,temporal_anomaly_block_count,temporal_anomaly_hour_history
0,2987000,0.375454,0.000000,0.375454,228,0
1,2987001,0.375454,0.000000,0.375454,228,1
2,2987002,0.742433,0.366979,0.375454,228,2
3,2987003,0.480542,0.105088,0.375454,228,3
4,2987004,0.471785,0.096331,0.375454,228,4
5,2987005,0.532697,0.157243,0.375454,228,5
6,2987006,8.615733,8.240279,0.375454,228,6
7,2987007,8.741457,8.366003,0.375454,228,7
8,2987008,1.102112,0.726658,0.375454,228,8
9,2987009,0.507056,0.131602,0.375454,228,9


### Aynı ders, üçüncü kez: sığ geçmiş = kararsız istatistik

En yüksek skorlu işlemlere bakalım; `temporal_anomaly_hour_history` (bu saat diliminin o ana kadar kaç işlem gördüğü) sütununa dikkat.

In [18]:
temporal_anomaly.sort_values("temporal_anomaly_score", ascending=False).head(10)

,TransactionID,temporal_anomaly_score,temporal_anomaly_amount_component,temporal_anomaly_burst_component,temporal_anomaly_block_count,temporal_anomaly_hour_history
1266,2988266,495.737411,493.110556,2.626855,148,2
611,2987611,442.377897,441.871028,0.506870,134,2
274336,3261336,146.232077,145.787172,0.444905,244,18772
274339,3261339,100.287041,99.842136,0.444905,244,18775
442,2987442,69.797514,69.062973,0.734542,180,13
4789,2991789,30.006555,27.564115,2.442440,354,21
464305,3451305,27.389386,27.389386,0.000000,102,16831
464347,3451347,26.823559,26.823559,0.000000,102,16873
296021,3283021,26.639112,26.639112,0.000000,207,20835
248413,3235413,25.917706,25.917706,0.000000,154,10903


En tepedeki iki satırın `hour_history`'si sadece **2**: o saat diliminin gördüğü ilk birkaç işlemden biri. Araştırdık: bu iki işlemden önceki 2 işlem neredeyse özdeş tutarlıydı (108,50 / 107,95), bu yüzden standart sapma neredeyse sıfıra çöktü ve 3. işlemin (300 TL, aslında pek de aşırı olmayan bir tutar) z-score'u 493'e fırladı. Bu, `column_anomaly`'nin `scored_count`'ı ve `entity_anomaly`'nin `history_depth`'i ile **birebir aynı mekanizma**; Case 4'ün dört katmanında da tekrarlayan bir prensip: nedensel ("so far") bir istatistik, az örnekle beslendiğinde kararsızlaşıyor.

**Bir fark var:** buradaki etkinin kapsamı çok daha dar. `entity_anomaly`'de medyan geçmiş derinliği sadece 7 işlemdi (13.553 kart arasında dağılmış); burada sadece 24 saat dilimi var, her biri saniyeler içinde onlarca-yüzlerce gözlem biriktiriyor; istikrarsızlık sadece her saat diliminin **ilk birkaç** işlemiyle sınırlı, veri setinin genelini etkilemiyor.

### Betimleyici kontrol: bulgular bu kez daha az net, dürüstçe raporlanıyor

In [19]:
isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()
check = temporal_anomaly.merge(isfraud, on="TransactionID")

check["score_bucket"] = pd.qcut(check["temporal_anomaly_score"], 5, labels=["1", "2", "3", "4", "5"], duplicates="drop")
print("kombine skora göre fraud oranı (%):")
print((check.groupby("score_bucket", observed=True)["isFraud"].mean() * 100).round(2))

kombine skora göre fraud oranı (%):
score_bucket
1    3.28
2    2.98
3    4.23
4    3.80
5    3.20
Name: isFraud, dtype: float64


Diğer üç katmanın aksine, burada net bir eğilim yok: %3,28 → %2,98 → %4,23 → %3,80 → %3,20, düzensiz. Muhtemelen tutar bileşenindeki sığ-geçmiş gürültüsü ile burst bileşeninin (aşağıda görülecek) fraud'la ters ilişkisi birbirini iptal ediyor.

In [20]:
check["burst_bucket"] = pd.cut(
    check["temporal_anomaly_burst_component"], bins=[-0.01, 0, 1, 2, 3, 100], labels=["0", "0-1", "1-2", "2-3", "3+"]
)
check.groupby("burst_bucket", observed=True)["isFraud"].agg(fraud_rate_pct=lambda s: s.mean() * 100, count="count")

,fraud_rate_pct,count
burst_bucket,,
0,3.703114,295049
0-1,3.713000,180878
1-2,3.086632,58154
2-3,2.481547,31432
3+,1.782075,25027


**Beklenmedik ve dürüstçe raporlanması gereken bir sonuç: burst arttıkça fraud oranı düşüyor** (%3,70 → %1,78, burst=0'dan 3+'ya). Bunun için bir varsayım kurmuyoruz ama Case 1'in bölüm 6'daki bulgusuyla tutarlı bir açıklama var: fraud oranı en yüksek saatler, **hacmin en düşük** olduğu saatlerdi (04:00-09:00). Burst tanımı gereği yoğun saatlerde (zaten yüksek hacimli) daha sık tetikleniyor; yani "patlama" burada çoğunlukla meşru yoğun alışveriş saatlerini yakalıyor, saldırı dalgalarını değil. Bu, tasarımın hatalı olduğu anlamına gelmiyor; burst hâlâ gerçek bir sinyali (beklenmedik hacim sapması) ölçüyor, sadece bu veri setinde fraud'la aynı yönde hareket etmiyor.

---

**Durum:** Case 4'ün 4. ve son katmanı (temporal anomaly) tamamlandı; dört katman da bitti.

## 5. Farklı Anomali Skorlarını Ayrı Ayrı Üretme

Case brief'in son maddesi, dört katmanı **tek bir sayıya indirmeden** bir araya getirmeyi istiyor; bu, sadece bir formalite değil, case brief'in kendi "tek modele bağlı kalmadan" ilkesinin doğrudan sonucu: dört bağımsız bakış açısını tek bir skorda eritmek, onları ayrı ayrı inşa etmenin bütün amacını ortadan kaldırır. `src/services/anomaly/combined.py` yeni bir hesaplama yapmıyor; sadece dört katmanın çıktısını `TransactionID` üzerinden birleştiriyor, her skor kendi kolonunda ayrı ayrı duruyor.

In [21]:
from src.services.anomaly.combined import compute_all_anomaly_scores, PRIMARY_SCORE_COLUMNS

all_scores = compute_all_anomaly_scores(parquet_path)
print(f"şekil: {all_scores.shape}  ({len(all_scores.columns)-1} skor/detay kolonu)")
all_scores[["TransactionID"] + PRIMARY_SCORE_COLUMNS].head(10)

şekil: (590540, 19)  (18 skor/detay kolonu)


,TransactionID,column_anomaly_mean_abs_zscore,multivariate_mahalanobis_distance,multivariate_isolation_forest_score,entity_anomaly_score,temporal_anomaly_score
0,2987000,0.226486,1.092386,-0.109392,0.0,0.375454
1,2987001,0.970291,1.358261,-0.095853,0.0,0.375454
2,2987002,0.289373,1.147110,-0.115453,0.0,0.742433
3,2987003,0.741222,1.769160,-0.052951,0.0,0.480542
4,2987004,0.688581,0.996790,-0.115155,0.0,0.471785
5,2987005,0.367379,1.240191,-0.111240,0.0,0.532697
6,2987006,0.579943,1.205895,-0.090468,0.0,8.615733
7,2987007,1.380759,2.036900,-0.000230,0.0,8.741457
8,2987008,0.993446,1.924905,-0.036925,0.0,1.102112
9,2987009,0.991504,0.751184,-0.049044,0.0,0.507056


### Neden ayrı tutmak önemli: beş skor birbirini ne kadar tekrarlıyor?

In [22]:
all_scores[PRIMARY_SCORE_COLUMNS].corr(method="spearman").round(2)

,column_anomaly_mean_abs_zscore,multivariate_mahalanobis_distance,multivariate_isolation_forest_score,entity_anomaly_score,temporal_anomaly_score
column_anomaly_mean_abs_zscore,1.00,0.70,0.75,0.21,0.19
multivariate_mahalanobis_distance,0.70,1.00,0.76,0.28,0.25
multivariate_isolation_forest_score,0.75,0.76,1.00,0.18,0.14
entity_anomaly_score,0.21,0.28,0.18,1.00,0.30
temporal_anomaly_score,0.19,0.25,0.14,0.30,1.00


Column ve multivariate skorları birbiriyle güçlü ilişkili (0,70-0,75; beklenen, çünkü multivariate'in kullandığı 3 kolon column anomaly'nin sinyalinin büyük kısmını da oluşturuyordu). Ama entity ve temporal, diğerleriyle ve birbirleriyle **zayıf** ilişkili (0,14-0,30); gerçekten farklı bir şey ölçüyorlar.

In [23]:
top1_masks = {col: all_scores[col] >= all_scores[col].quantile(0.99) for col in PRIMARY_SCORE_COLUMNS}
overlap = pd.DataFrame(top1_masks).sum(axis=1)
overlap.value_counts().sort_index().to_frame(name="işlem sayısı")

,işlem sayısı
0,570052
1,13314
2,5359
3,1766
4,45
5,4


**En güçlü kanıt burada:** her katmanın kendi en yüksek %1'lik dilimini (5.905'er işlem) alıp kaç katmanda ortak olduklarına baktık. 590.540 işlemin sadece **4'ü** beş katmanın **hepsinde** top-%1'de. 13.314 işlem sadece **bir** katmanda yakalanıyor; yani sadece tek bir katman kullansaydık, diğer dördünün gördüğü şüpheli işlemlerin büyük kısmını kaçırırdık. Case 4'ün "tek modele bağlı kalmama" tezinin sayısal kanıtı bu tablo.

---

**Case 4 tamamen bitti:** beş madde de tamamlandı, column, multivariate, entity, temporal katmanları ve bunların ayrı ayrı tutulduğu birleşik çıktı.

## 6. ROC-AUC: dört katmanın betimleyici karşılaştırması

Yukarıdaki kova tabloları her katmanın fraud oranıyla ilişkisini gösterdi, ama bazıları (entity,
temporal) "düzensiz" olarak raporlandı: kova sınırlarının kendisi bir miktar gürültü katıyor olabilir.
ROC-AUC, eşiksiz bir ayırt edicilik ölçüsü: 0,5 rastgeleye eşit, 1,0 mükemmel sıralama. Bu, IEEE-CIS
Kaggle yarışmasının resmi değerlendirme metriği; burada da aynı disiplinle kullanılıyor: `isFraud`
sadece dört katman zaten hesaplandıktan SONRA, betimleyici bir karşılaştırma için okunuyor, hiçbir
katmanın hesaplanma şeklini etkilemiyor.

In [24]:
from src.services.evaluation.roc import compare_auc

isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()
scored = all_scores.merge(isfraud, on="TransactionID")

auc_table = compare_auc(scored, PRIMARY_SCORE_COLUMNS)
auc_table

,score_column,auc
0,multivariate_isolation_forest_score,0.626861
1,multivariate_mahalanobis_distance,0.602329
2,column_anomaly_mean_abs_zscore,0.600926
3,entity_anomaly_score,0.552138
4,temporal_anomaly_score,0.508790


**AUC sıralaması, kova tablolarındaki izlenimi doğruluyor ve netleştiriyor:**
Isolation Forest (0,627) ve Mahalanobis (0,602) en ayırt edici iki katman; column da yakın
(0,601). Entity (0,552) zayıf ama rastgeleden farklı; temporal (0,509) neredeyse tam rastgele,
0,5'e o kadar yakın ki burst bileşeninin fraud'la ters ilişkili olması ile hacim/temel bileşenin
düz ilişkisi birbirini pratikte iptal ediyor demek. Bu, kova tablosundaki "düzensiz" gözleminin tek
bir sayıyla teyidi: temporal katmanı tek başına neredeyse hiçbir ayırt edicilik taşımıyor, ama yine
de tutuluyor çünkü Madde 5'in top-%1 örtüşme analizinde diğer katmanların kaçırdığı bazı işlemleri
yakalıyor: düşük AUC, "işe yaramaz" anlamına gelmiyor, "tek başına zayıf ama toplamda tamamlayıcı"
anlamına geliyor.

## Sonuç: Case 4'ün dört anomali katmanı

| # | Katman | Dosya | Bakış açısı | Fraud ile ilişki | AUC |
|---|---|---|---|---|---|
| 1 | Column | `anomaly/column_anomaly.py` | Her sayısal kolon, global popülasyona göre (MAD z-score) | Güçlü, çoğunlukla monotonik (%1,71→%5,34); ama gerçek sinyal sadece 8/96 kolondan; 88'i MAD=0 | 0,601 |
| 2 | Multivariate | `anomaly/multivariate_anomaly.py` | 3 tam-dolu kolon birlikte (Mahalanobis + Isolation Forest) | Her iki yöntem de güçlü ve monotonik (%1,8-2,0→%4,9-5,9); yöntemler arası Spearman=0,76 | 0,627 / 0,602 |
| 3 | Entity | `anomaly/entity_anomaly.py` | Aynı `card1`'in kendi geçmişine göre (Case 3'ün feature'ları yeniden kullanılıyor) | Zayıf/düzensiz eğilim (%2,9-4,8); soğuk-başlangıç kartları beklenenin aksine daha düşük fraud oranı | 0,552 |
| 4 | Temporal | `anomaly/temporal_anomaly.py` | Aynı saat diliminin kendi geçmişine göre + hacim patlaması | Kombine skor düzensiz; burst bileşeni fraud'la **ters** ilişkili (yoğun saatler = meşru trafik) | 0,509 |

**Katmanlar arası tekrarlayan iki prensip:**

1. **Nedensel ("so far") istatistikler, az örnekle beslendiğinde kararsızlaşır.** Bu, üç ayrı katmanda (column'da `scored_count`, entity'de `history_depth`, temporal'da `hour_history`) bağımsız olarak keşfedildi ve her seferinde şeffaf bir "bu skor kaç gözleme dayanıyor" kolonuyla raporlandı; gizlenmedi, ölçüldü.
2. **Hiçbir katman `isFraud`'u hesaplamada kullanmadı.** Betimleyici kontroller bazen bekleneni doğruladı (multivariate, column), bazen tam tersini gösterdi (entity'nin soğuk-başlangıç bulgusu, temporal'ın burst-fraud ters ilişkisi); ikisi de olduğu gibi raporlandı, tasarım etikete göre ayarlanmadı.

**Madde 5'in bulgusu bunu sayısallaştırdı:** 590.540 işlemin sadece 4'ü beş skorun hepsinde top-%1'de; 13.314'ü sadece bir katmanda yakalanıyor. Tek katmana güvenmek diğer dördünün gördüğünü kaçırmak demek.

**Case 4'ün asıl tezi doğrulandı:** dört katman farklı sonuçlar veriyor (bazen aynı yönde, bazen zıt yönde); tek bir modele güvenmek yerine bu çeşitliliğin kendisi, sistemin neden çok katmanlı tasarlandığının kanıtı.